# DL · Encuentro 3 · Secuencias, atención y Transformer

Secuencias, atención y contexto

**Tu copia personal:** en Colab elige **Archivo → Guardar una copia en Drive**. Conserva esa copia para los tres temas del encuentro.

Trabaja en CPU. Estas prácticas no requieren contratar GPU, claves de API ni servicios de pago. La primera preparación puede descargar bibliotecas ausentes; las prácticas de detección descargan los pesos del modelo si no están disponibles.

## Tu recorrido

1. [El orden puede cambiar el significado](#scrollTo=tema-dl-secuencias)
2. [Relacionar elementos con contexto](#scrollTo=tema-dl-atencion)
3. [Atención más información de orden](#scrollTo=tema-dl-transformers)

Cada tema reúne **idea → caso explicado → preparación → ejecución → lectura y comprobación**. Puedes ejecutar todo el cuaderno o empezar por un tema: cada uno vuelve a preparar sus datos y herramientas. No mezcles las cifras de conjuntos diferentes.

Los datos inventados se identifican como simulaciones. Las fotos de detección son demostraciones y no incluyen una evaluación anotada de sus objetos. Las comprobaciones son voluntarias y sin nota.


## Tema 1 · Pronosticar una lectura con LSTM

**Lo que vas a poder hacer:** ¿Una red puede anticipar la siguiente lectura mejor que repetir la última?

Usaremos una serie artificial de 240 lecturas sin unidades físicas. Cada entrada contiene las 12 lecturas anteriores; la respuesta buscada es la lectura que sigue. El corte temporal reserva los últimos 60 instantes para comprobar, uno por uno.

### Predice antes de ejecutar

Dibuja una subida. Si el pronóstico repite la última lectura disponible, ¿quedará por encima o por debajo de la siguiente si la subida continúa? Escribe tu predicción sin mirar la salida.

### Ejecuta y compara

1. Ejecuta las dos celdas de preparación de este tema y después «Una LSTM real». Espera al gráfico y a los dos números MAE.
2. Elige el mismo instante en las tres líneas: Real simulado, LSTM y Último valor. Compara quién queda más cerca en una subida y en una bajada. Después compara los dos MAE; menor significa menor distancia media en esta reserva, no acierto garantizado en cada instante.

### Explica con tus palabras

«En este instante la entrada es el pasado disponible; la respuesta real es…; el pronóstico de… queda más cerca porque…».

**Cómo conecta con el aula:** La regla de mitades de la web muestra cómo actualizar un resumen. Aquí una LSTM (memoria de largo y corto plazo) sí aprende parámetros y una capa de salida produce un pronóstico. Son datos y resultados diferentes.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Conserva primero datos, semilla y entrenamiento. Cambia el instante que observas y compara los mismos tres valores. No ejecutes de nuevo solo para buscar un resultado que te guste.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Pon una pregunta junto al gráfico

Ejecuta la celda. Cuando termine, elige un instante de subida y compara las tres lecturas en ese mismo punto. Las doce lecturas anteriores son la entrada; la siguiente es la respuesta que queremos anticipar. La LSTM actualiza un estado y una capa de salida lo convierte en pronóstico.


In [ ]:
# Una LSTM real para una secuencia pequeña y artificial.
r=np.random.default_rng(SEMILLA);tiempo=np.arange(240);serie=np.sin(tiempo*.12)+.12*r.normal(size=len(tiempo))
VENTANA=12;CORTE=180
entradas=np.array([serie[i-VENTANA:i] for i in range(VENTANA,len(serie))],dtype='float32')
objetivos=serie[VENTANA:].astype('float32');instante=tiempo[VENTANA:]
train=instante<CORTE;test=~train
xt=torch.tensor(entradas[train,:,None]);yt=torch.tensor(objetivos[train,None]);xe=torch.tensor(entradas[test,:,None])
class PredictorLSTM(nn.Module):
    def __init__(self):
        super().__init__();self.memoria=nn.LSTM(1,12,batch_first=True);self.salida=nn.Linear(12,1)
    def forward(self,x):
        secuencia,_=self.memoria(x);return self.salida(secuencia[:,-1])
torch.manual_seed(SEMILLA);lstm=PredictorLSTM();opt=torch.optim.Adam(lstm.parameters(),lr=.02)
for epoca in range(90):
    opt.zero_grad();error=nn.functional.mse_loss(lstm(xt),yt);error.backward();opt.step()
lstm.eval()
with torch.no_grad():p=lstm(xe).numpy().ravel()
baseline=entradas[test,-1]
print('MAE LSTM:',round(mean_absolute_error(objetivos[test],p),3),'· MAE último valor:',round(mean_absolute_error(objetivos[test],baseline),3))
plt.plot(instante[test],objetivos[test],label='Real simulado',color=GRIS);plt.plot(instante[test],p,label='LSTM',color=AZUL);plt.plot(instante[test],baseline,label='Último valor',color=VERDE,alpha=.6)
plt.legend();plt.xlabel('Tiempo');plt.ylabel('Lectura simulada (sin unidades)');plt.title('Pronóstico de un paso usando la ventana disponible');mostrar('11-lstm')
print('Se evalúa un paso hacia adelante; no es un pronóstico recursivo de toda la serie.')


### Después de ejecutar: construye el error paso a paso

Primero compara las líneas **Real simulado**, **LSTM** y **Último valor** en el mismo instante. Estar más cerca de la línea real significa cometer un error menor en ese instante.

**MAE** significa error absoluto medio. Se construye así, con un ejemplo independiente del gráfico:

1. Una respuesta real de 1 y un pronóstico de 0,7 dejan una diferencia de 1−0,7=0,3.
2. Tomamos su magnitud: |0,3|=0,3. Si el pronóstico fuera 1,3, |1−1,3|=0,3 también. Los errores por encima y por debajo no se cancelan.
3. Si tres errores fueran 0,3; 0,1 y 0,2, los sumamos: 0,3+0,1+0,2=0,6.
4. Repartimos entre los tres casos: 0,6÷3=0,2. Esa es la distancia media de ese ejemplo, no el resultado de la LSTM.

Ahora lee los **dos MAE que imprime tu ejecución**. Se calcularon sobre los mismos instantes reservados: menor MAE significa menor distancia media en esa reserva. El promedio puede mejorar aunque un punto concreto empeore. Los valores no tienen unidades físicas ni son porcentajes.

Son 60 comprobaciones de un paso usando el pasado disponible en cada instante, no un pronóstico recursivo de los 60 pasos a partir de una sola ventana. Si eliges parámetros mirando repetidamente esta reserva, pasa a ser validación y necesitas otra prueba final.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Elige el mismo instante en las tres líneas: Real simulado, LSTM y Último valor. Compara quién queda más cerca en una subida y en una bajada. Después compara los dos MAE; menor significa menor distancia media en esta reserva, no acierto garantizado en cada instante.

**Prueba pequeña:** Conserva primero datos, semilla y entrenamiento. Cambia el instante que observas y compara los mismos tres valores. No ejecutes de nuevo solo para buscar un resultado que te guste.

**Pregunta:** ¿Una red puede anticipar la siguiente lectura mejor que repetir la última?

**Respuesta razonada:** El estado resume lo leído; la capa de salida produce el pronóstico. Un MAE menor significa menor distancia media sobre la misma reserva, sin garantizar que el modelo gane en cada instante. Usa los dos números que obtuviste para justificar tu comparación.

**Registra en tu copia:** «En este instante la entrada es el pasado disponible; la respuesta real es…; el pronóstico de… queda más cerca porque…». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-secuencias)

[Siguiente tema: Relacionar elementos con contexto](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-atencion)


## Tema 2 · Mezclar dos registros con atención

**Lo que vas a poder hacer:** ¿Cómo cambia una mezcla si consultamos de otra forma los mismos dos registros?

A contiene 10 cajas ficticias y B contiene 20. Las claves K son [1,0] y [0,1]. Las consultas Q son [1,0], [0,1] y [0,0]. Los valores V que se mezclan son 10 y 20. Todos estos números están dados; aquí no se entrena una red.

### Predice antes de ejecutar

Para la consulta [0,0], ambas fuentes recibirán el mismo peso. Calcula por tu cuenta mitad de 10 + mitad de 20. Para [1,0], ¿esperas una mezcla más cerca de 10 o de 20?

### Ejecuta y compara

1. Ejecuta la preparación de este tema y la celda de las dos fuentes. No necesitas ejecutar la LSTM anterior.
2. Compara las filas [1,0] y [0,0] del mapa: cuánto participa A, cuánto B y cuánto suman. Sigue esas participaciones hasta las barras de mezcla. Comprueba con tus números la fila neutra antes de leer las demás.

### Explica con tus palabras

«Cambió Q; K y V se conservaron. La fuente… recibió más participación y por eso la mezcla se acercó a…».

**Cómo conecta con el aula:** Atención compara Q con K, calcula participaciones y mezcla V. Es una operación que luego puede formar parte de un Transformer; la LSTM no se convierte en Transformer al ejecutar esta celda.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Conserva claves y valores. El código ya calcula tres consultas para que compares únicamente ese cambio.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Tres nombres para seguir la cuenta

**Q, consulta:** lista de números que determina qué comparamos. **K, claves:** listas de cada fuente con las que se compara Q. **V, valores:** cantidades que se mezclan después. Una lista de números se llama vector; [1,0] tiene dos componentes, no es una fracción.

**Softmax** convierte las comparaciones en participaciones positivas cuya suma es 1. Verás sus operaciones con números en el paso siguiente.


### De comparar a mezclar: una sola consulta

Sigue Q=[1,0], K de A=[1,0], K de B=[0,1], V de A=10 y V de B=20. Son números dados para enseñar la operación.

| Paso | Sustituye los números | Qué significa |
| --- | --- | --- |
| Comparar con A | 1×1 + 0×0 = 1 | Multiplicar componentes del mismo lugar y sumar se llama producto punto. |
| Comparar con B | 1×0 + 0×1 = 0 | La misma consulta se compara con la otra clave. |
| Ajustar la escala | 1÷√2≈0,7071; 0÷√2=0 | d=2 es el número de componentes de cada clave. √2≈1,4142. Este ajuste forma parte de esta atención. |
| Preparar una cuenta estable | 0,7071−0,7071=0; 0−0,7071≈−0,7071 | Restamos el mayor a ambos. Evita números enormes al usar exponenciales y conserva el reparto final. |
| Convertir a cantidades positivas | exp(0)=1; exp(−0,7071)≈0,4931 | exp es la función exponencial; usa calculadora o el programa, no necesitas resolverla mentalmente. |
| Sumar las partes | 1+0,4931≈1,4931 | Total antes de repartir. |
| Dividir cada parte por el total | A: 1÷1,4931≈0,6698; B: 0,4931÷1,4931≈0,3302 | Participaciones aproximadas del 66,98 % y 33,02 %; suman 100 %. |
| Calcular aportes y sumarlos | 0,6698×10 + 0,3302×20≈13,302 | Cada participación multiplica su valor; la suma es el contexto o mezcla. |

El código conserva más decimales que esta tabla. La mezcla queda más cerca de 10 porque A participa más. Estos pesos cambian con Q; no son parámetros entrenados ni demuestran una causa.


### Ahora comprueba tu predicción

Ejecuta la celda y busca primero Q=[0,0]. ¿Coincide con la mezcla que calculaste antes? Después compara Q=[1,0] con Q=[0,1] sin cambiar las claves ni los valores.


In [ ]:
# Dos registros como en el aula: mezclamos cantidades con pesos dados por la comparación.
fuentes=['A: 10','B: 20']
consultas=['Q=[1,0]','Q=[0,1]','Q=[0,0]']
Q=np.array([[1.,0.],[0.,1.],[0.,0.]])
K=np.array([[1.,0.],[0.,1.]])
V=np.array([[10.],[20.]])
puntuaciones=Q@K.T/np.sqrt(2)
exp=np.exp(puntuaciones-puntuaciones.max(axis=1,keepdims=True))
atencion=exp/exp.sum(axis=1,keepdims=True)
mezclas=atencion@V
assert np.allclose(atencion.sum(axis=1),1)
assert np.allclose(atencion[2],[.5,.5]) and np.isclose(mezclas[2,0],15)
for consulta,pesos,mezcla in zip(consultas,atencion,mezclas[:,0]):
    print(consulta,'· participación A/B:',np.round(pesos,4),'· mezcla:',round(float(mezcla),3))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(atencion,cmap='Blues',vmin=0,vmax=1)
axs[0].set_xticks(range(2),fuentes);axs[0].set_yticks(range(3),consultas)
for i in range(3):
    for j in range(2):axs[0].text(j,i,f'{atencion[i,j]*100:.1f} %',ha='center',va='center',color='black')
axs[0].set(title='Cuánto participa cada fuente',xlabel='Fuente',ylabel='Consulta dada')
axs[1].bar(consultas,mezclas[:,0],color=[AZUL,VERDE,GRIS]);axs[1].set(title='Resultado de cada mezcla',ylabel='Valor mezclado',ylim=(0,22))
axs[1].axhline(10,ls=':',color='gray');axs[1].axhline(20,ls=':',color='gray')
mostrar('12-atencion')


### Después de ejecutar: une la fila con la barra

En Q=[0,0], la participación es 0,5 y 0,5; los aportes son 0,5×10=5 y 0,5×20=10; la mezcla es 5+10=15. Q=[1,0] produce una mezcla cercana a 13,302 y Q=[0,1] una cercana a 16,698.

Cada fila del mapa debe sumar 100 %. Cada barra representa una mezcla, no un porcentaje ni una cantidad de objetos detectados. Aquí Q, K y V estaban dados y no hubo entrenamiento. El tema siguiente usa atención dentro de otra arquitectura y entrena una tarea diferente.


### Lee, comprueba y explica

**Qué debes poder comprobar:** Compara las filas [1,0] y [0,0] del mapa: cuánto participa A, cuánto B y cuánto suman. Sigue esas participaciones hasta las barras de mezcla. Comprueba con tus números la fila neutra antes de leer las demás.

**Prueba pequeña:** Conserva claves y valores. El código ya calcula tres consultas para que compares únicamente ese cambio.

**Pregunta:** ¿Cómo cambia una mezcla si consultamos de otra forma los mismos dos registros?

**Respuesta razonada:** La consulta neutra da participaciones 0,5/0,5 y mezcla 15. La primera consulta acerca la mezcla a 10 y la segunda a 20. Cambian los pesos de atención; los valores y los parámetros dados se conservan.

**Registra en tu copia:** «Cambió Q; K y V se conservaron. La fuente… recibió más participación y por eso la mezcla se acercó a…». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-atencion)

[Siguiente tema: Atención más información de orden](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transformers)


## Tema 3 · Comprobar el orden con un Transformer

**Lo que vas a poder hacer:** ¿La respuesta de una red coincide con una regla que tú puedes comprobar?

La tarea pregunta únicamente «¿el último número es menor que el primero?». Clase 1 significa sí y clase 0 significa no. Se entrena un Transformer pequeño con secuencias artificiales de cuatro números; los patrones de prueba no se comparten con entrenamiento.

### Predice antes de ejecutar

Resuelve estas cuatro filas mirando solo los extremos: [1,2,3,4], [4,3,2,1], [2,7,1,3] y [3,1,7,2]. Anota una etiqueta 0 o 1 por fila antes de ejecutar. Una subida intermedia no cambia la pregunta.

### Ejecuta y compara

1. Ejecuta la preparación de este tema y el Transformer. Espera a que termine el entrenamiento y aparezcan las cuatro comparaciones.
2. En cada fila, separa «respuesta conocida» de «respuesta de la red». Marca coincidencia o error. La puntuación de clase 1 es la preferencia calculada para sí; no es la etiqueta correcta. El acierto en prueba resume otros patrones reservados.

### Explica con tus palabras

«La referencia de esta fila es… porque comparé… con…; la red respondió…; por tanto aquí acertó/falló».

**Cómo conecta con el aula:** El modelo representa números, añade información de posición, relaciona elementos mediante atención y calcula dos puntuaciones de clase. La web usa puntuaciones dadas para aprender a leerlas; aquí sí se calculan con parámetros entrenados.

**Antes de ejecutar:** este tema incluye su preparación. Ejecuta sus celdas en orden; no requiere los temas anteriores. Si repites el tema desde cero, comienza por su preparación.

**Qué cambiar y qué conservar:** Conserva primero los cuatro casos y compara sus respuestas. No confundas «último menor que primero» con «todos los pasos disminuyen». En esta prueba se excluyen extremos iguales.


In [ ]:
# Preparación 1: instalar únicamente bibliotecas que falten.
import importlib.util, subprocess, sys
paquetes = {'numpy':'numpy', 'pandas':'pandas', 'matplotlib':'matplotlib',
            'sklearn':'scikit-learn'}
for modulo, paquete in paquetes.items():
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                           '--index-url', 'https://download.pytorch.org/whl/cpu'])
print('Bibliotecas listas. No necesitas activar GPU.')

# Las extensiones se preparan solo cuando se abre su bloque.
def preparar_extension(modulo, paquete):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])


In [ ]:
# Preparación 2: herramientas compartidas.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
from sklearn.datasets import load_digits
from IPython.display import display

# Estas opciones mantienen fuera de la ruta principal el material de ampliación.
# Déjalas en False para los cuatro encuentros de DL.
EJECUTAR_REPASO_ML = False
EJECUTAR_TEXTO = False
EJECUTAR_GENERACION = False

SEMILLA = 42
np.random.seed(SEMILLA); torch.manual_seed(SEMILLA); torch.set_num_threads(1)
AZUL, VERDE, GRIS = '#0075ad', '#198a87', '#9eb4c7'
plt.rcParams.update({'figure.figsize':(9,4.5), 'font.size':11,
                     'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.titleweight':'bold', 'axes.grid':False})
# En Colab puedes descargar los PNG desde el panel Archivos.
CARPETA = Path('figuras_utp'); CARPETA.mkdir(exist_ok=True)
def mostrar(nombre):
    plt.tight_layout()
    plt.savefig(CARPETA / (nombre + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def datos_fincas(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Humedad_pct':r.uniform(20,70,n),
                      'Temperatura_C':r.uniform(18,35,n),
                      'Lluvia_mm':r.uniform(0,100,n)})
    y = (2 + .07*X.Humedad_pct + .012*X.Lluvia_mm
         - .025*(X.Temperatura_C-24)**2 + r.normal(0,.3,n))
    return X, y.to_numpy()

def datos_cafe(n=240, semilla=SEMILLA):
    r = np.random.default_rng(semilla)
    X = pd.DataFrame({'Uniformidad_color':r.uniform(0,1,n),
                      'Proporcion_manchas':r.uniform(0,1,n)})
    puntuacion = 2*X.Uniformidad_color-X.Proporcion_manchas-.5+r.normal(0,.15,n)
    return X, (puntuacion > 0).astype(int).to_numpy() # 1=sano, 0=defectuoso
print('Semilla:',SEMILLA,'· PyTorch:',torch.__version__)
print('Todas las cifras agrícolas son simuladas.')


### Conserva tus cuatro etiquetas junto a la salida

Antes de ejecutar, anota tus etiquetas 0/1 de las cuatro secuencias de la introducción. El programa entrena una red; tus etiquetas manuales sirven para comprobar sus respuestas, no para sustituirlas.


In [ ]:
# Transformer real, pequeño, entrenado para una tarea de orden.
r=np.random.default_rng(SEMILLA);seq=r.integers(1,8,size=(700,4));seq=seq[seq[:,0]!=seq[:,-1]]
seq=np.unique(seq,axis=0) # reserva de patrones distintos, no solo de filas
y=(seq[:,0]>seq[:,-1]).astype('int64')
Xtr,Xte,ytr,yte=train_test_split(seq,y,test_size=.25,stratify=y,random_state=SEMILLA)
compartidas=set(map(tuple,Xtr)) & set(map(tuple,Xte))
assert not compartidas, 'Entrenamiento y prueba comparten una secuencia.'
print('Secuencias distintas:',len(seq),'· entrenamiento:',len(Xtr),'· prueba:',len(Xte))
print('Clases [0, 1] en entrenamiento:',np.bincount(ytr),'· en prueba:',np.bincount(yte))
class OrdenTransformer(nn.Module):
    def __init__(self):
        super().__init__();self.emb=nn.Embedding(8,16);self.pos=nn.Parameter(torch.randn(1,4,16)*.1)
        capa=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(capa,num_layers=1,enable_nested_tensor=False);self.salida=nn.Linear(4*16,2)
    def forward(self,x):
        # La salida por posiciones también conserva orden; no aislamos el efecto de self.pos.
        h=self.encoder(self.emb(x)+self.pos);return self.salida(h.flatten(1))
torch.manual_seed(SEMILLA);transformer=OrdenTransformer();opt=torch.optim.Adam(transformer.parameters(),lr=.005)
loader=DataLoader(TensorDataset(torch.tensor(Xtr),torch.tensor(ytr)),batch_size=64,shuffle=True)
hist=[]
for epoca in range(45):
    transformer.train();suma=0
    for bx,by in loader:
        opt.zero_grad();error=nn.functional.cross_entropy(transformer(bx),by);error.backward();opt.step();suma+=error.item()*len(bx)
    hist.append(suma/len(Xtr))
transformer.eval()
with torch.no_grad():
    pred=transformer(torch.tensor(Xte)).argmax(1).numpy()
    ejemplos=torch.tensor([[1,2,3,4],[4,3,2,1],[2,7,1,3],[3,1,7,2]]);prob=torch.softmax(transformer(ejemplos),dim=1).numpy()
print('Accuracy en prueba:',round(accuracy_score(yte,pred),3))
esperadas=(ejemplos[:,0]>ejemplos[:,-1]).numpy().astype(int)
calculadas=prob.argmax(axis=1)
for secuencia,esperada,calculada,p1 in zip(ejemplos.tolist(),esperadas,calculadas,prob[:,1]):
    print(secuencia,'· respuesta conocida:',int(esperada),'· respuesta de la red:',int(calculada),'· puntuación clase 1:',round(float(p1),3))
print('Comparar extremos no pregunta si toda la secuencia baja. Las respuestas de la red pueden fallar.')
plt.plot(hist,color=AZUL);plt.xlabel('Época');plt.ylabel('Pérdida');plt.title('Pérdida de entrenamiento en la tarea de orden');mostrar('12-transformer')


### Después de ejecutar: comprueba cada respuesta

Las cuatro referencias son **0, 1, 0, 1**: en cada fila se compara únicamente el último número con el primero. Una subida intermedia no cambia la tarea. Busca para cada fila la «respuesta de la red» y marca si coincide; una puntuación alta también puede acompañar un error.

**Cómo leer la puntuación:** el modelo calcula dos valores y softmax los transforma en dos participaciones que suman 1. El programa imprime la de clase 1. Por ejemplo, si imprimiera 0,80 para clase 1, la de clase 0 sería 1−0,80=0,20. Elegiría 1 porque 0,80 es mayor que 0,20. Es un ejemplo de lectura, no una salida prometida. El código elige la mayor; en empate exacto elige clase 0.

**Cómo leer «Accuracy en prueba»:** cuenta respuestas correctas y divide entre los casos reservados. Ejemplo independiente: 3 aciertos÷4 casos=0,75, equivalente al 75 %. El valor que imprime tu ejecución procede de los patrones reservados, no solo de estas cuatro filas. La curva de pérdida pertenece al entrenamiento y no es ese porcentaje.

El modelo del cuaderno representa cada número, añade posición, aplica un bloque con atención y otras transformaciones y calcula la salida. Su lectura final también conserva las posiciones; esta práctica no aísla el efecto de añadir posición. El resultado comprueba esta tarea artificial, no tareas de conversación ni lenguaje.


### Lee, comprueba y explica

**Qué debes poder comprobar:** En cada fila, separa «respuesta conocida» de «respuesta de la red». Marca coincidencia o error. La puntuación de clase 1 es la preferencia calculada para sí; no es la etiqueta correcta. El acierto en prueba resume otros patrones reservados.

**Prueba pequeña:** Conserva primero los cuatro casos y compara sus respuestas. No confundas «último menor que primero» con «todos los pasos disminuyen». En esta prueba se excluyen extremos iguales.

**Pregunta:** ¿La respuesta de una red coincide con una regla que tú puedes comprobar?

**Respuesta razonada:** Las referencias manuales son 0, 1, 0 y 1. Solo después de ejecutar sabes qué respondió esta red entrenada. Compara cada predicción con su referencia; su puntuación de clase 1 por sí sola no certifica que acertó.

**Registra en tu copia:** «La referencia de esta fila es… porque comparé… con…; la red respondió…; por tanto aquí acertó/falló». No es una entrega ni una calificación.

[Volver al tema del aula](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-transformers)

[Siguiente tema: De una foto a un registro](https://nexo-ia-registros-utp.niftyacorn2048.chatgpt.site/aula/#tema/dl-yolo)


## Cierra el encuentro con una explicación

1. **El orden puede cambiar el significado:** La media pierde información de orden. Este estado conserva parte de ella con una regla dada; todavía no pronostica la siguiente entrega.
2. **Relacionar elementos con contexto:** Comparar fuentes y mezclar su contenido son dos operaciones diferentes. Aquí se calcula atención; no se entrena un Transformer.
3. **Atención más información de orden:** Atención es una pieza del Transformer. Conocer una regla sencilla permite comprobar su respuesta, pero no demuestra que la red ya la aprendió.

En cada resultado identifica entrada, salida, unidad o significado, forma de comprobarlo y límite. Guarda tu copia personal. Colab no registra automáticamente tu avance en el aula.
